# 第 12 章：MoE 融合算子和性能分析 — 动手实验

## 小节概述

本小节是第 12 章的核心实践环节。你将在昇腾 NPU 上完成 MoE Router 融合算子
（`matmul → softmax → topk → renorm`，4 合 1）的**设计分析 + 工程全流程 + 性能归因**：

1. 环境检查（npu-smi / CANN 版本 / 工具链）
2. 基线参考实现与测试数据（FP32 参考 + 4 算子序列计时 + 8 组边界用例）
3. 融合设计分析（数据流 / HBM 流量 / UB 预算 / 融合判据）
4. 算子工程与源码走读（纯标量范式 / 多核连续块切分 / cache line 约束）
5. 编译 → 打包 → 部署
6. 正确性验证（单用例 + 8 用例一键回归）
7. 性能测量与归因分析（vs 4 算子基线）

> 所有命令均在本 Notebook 的 code cell 中执行，无需手动打开终端。
> 源码工程位于 `src/custom_op/`，通过 `cat` / `sed` / `find` 直接查看与讲解。


## 步骤 1：确认环境和目标平台

In [ ]:
import os
import subprocess

# 工作目录定位：保证从本 Notebook 所在章节目录执行
if not os.path.exists('src/custom_op'):
    for p in ['contrib/tutorials/data_structures_compute/12_moe_fused']:
        if os.path.exists(os.path.join(p, 'src/custom_op')):
            os.chdir(p)
            break
print('工作目录:', os.getcwd())

# CANN 路径探测（兼容 CANNLab 默认安装与自定义安装路径）
_cands = [os.environ.get('ASCEND_HOME_PATH', ''),
          '/usr/local/Ascend/ascend-toolkit/latest',
          os.path.expanduser('~/Ascend/cann-9.0.0')]
ASCEND_HOME = next((c for c in _cands if c and os.path.exists(os.path.join(c, 'set_env.sh'))),
                   '/usr/local/Ascend/ascend-toolkit/latest')
print(f'ASCEND_HOME_PATH = {ASCEND_HOME}')

def run(cmd, **kw):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout, end='')
    if r.stderr: print(r.stderr, end='')
    return r.returncode

# 1) NPU 设备
run('npu-smi info | head -18')
# 2) CANN 版本
run(f'cat {ASCEND_HOME}/version.cfg 2>/dev/null | head -3 || ls {ASCEND_HOME}')
# 3) 依赖工具
run('which cmake gcc python3')
print('环境检查完成。')

## 步骤 2：基线参考实现与测试数据

### 2.1 工具说明

- `tools/moe_ref.py`：FP32 参考实现（`matmul → softmax → topk → renorm`），
  含 CPU 自洽校验、CPU/NPU 一致性校验，以及 **4 算子序列（torch_npu）baseline 计时**；
- `tools/gen_test_data.py`：生成 8 组固定种子（seed=42）测试用例，覆盖边界形态。

8 组用例（N=token 数，D=隐层维度，E=专家数，K=Top-K）：

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr><th>用例</th><th>N</th><th>D</th><th>E</th><th>K</th><th>边界意义</th></tr>
  </thead>
  <tbody>
    <tr><td>case_128_512_16_2</td><td>128</td><td>512</td><td>16</td><td>2</td><td>标准形态</td></tr>
    <tr><td>case_100_512_16_2</td><td>100</td><td>512</td><td>16</td><td>2</td><td>N 非 32 倍数</td></tr>
    <tr><td>case_129_256_16_2</td><td>129</td><td>256</td><td>16</td><td>2</td><td>N 非 32 倍数且 &gt; 32</td></tr>
    <tr><td>case_16_256_8_2</td><td>16</td><td>256</td><td>8</td><td>2</td><td>最小规模（单核）</td></tr>
    <tr><td>case_256_1024_4_4</td><td>256</td><td>1024</td><td>4</td><td>4</td><td>E=4 小专家边界</td></tr>
    <tr><td>case_512_512_8_2</td><td>512</td><td>512</td><td>8</td><td>2</td><td>多核切分</td></tr>
    <tr><td>case_1024_1024_32_4</td><td>1024</td><td>1024</td><td>32</td><td>4</td><td>E=32 / K=4</td></tr>
    <tr><td>case_4096_2048_8_2</td><td>4096</td><td>2048</td><td>8</td><td>2</td><td>大 N / 大 D</td></tr>
  </tbody>
</table>

In [ ]:
# 查看参考实现（开头 60 行：接口定义与用法）
run('sed -n "1,60p" tools/moe_ref.py')

### 2.2 运行参考实现校验与基线计时

预期输出：`[cpu] self-consistency PASS`、`[npu] ... 一致率 100%`、三个形状的 4 算子
序列耗时（launch 开销主导，约 0.22~0.31 ms），最后 `ALL CHECKS PASSED`。

In [ ]:
# FP32 参考实现校验 + 4 算子序列（torch_npu）baseline 计时
run(f'source {ASCEND_HOME}/set_env.sh && python3 tools/moe_ref.py --npu --bench')

### 2.3 生成测试数据

每用例包含：`x_fp16.bin` / `wgate_fp16.bin`（FP16 输入）、`ref_topk_idx.npy` /
`ref_topk_weights.npy`（FP32 参考）、`meta.json`（形状元信息）。
固定 seed=42，重复运行结果幂等。

In [ ]:
# 生成 8 组测试数据（幂等）
run('python3 tools/gen_test_data.py')
run('ls data/ && du -sh data/')

## 步骤 3：融合设计分析

完整设计文档位于 `docs/design.md`（定稿版）。本步骤提炼四个关键设计结论。

### 3.1 融合收益：中间张量流量清零

记 N=token 数、D=隐层维度、E=专家数、K=Top-K，元素按 2 字节（FP16）计：

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr><th>中间张量</th><th>形状</th><th>Baseline（4 算子）</th><th>融合后</th></tr>
  </thead>
  <tbody>
    <tr><td>scores（写 + softmax 读）</td><td>[N, E]</td><td>2·N·E·2 B</td><td><b>0</b>（UB）</td></tr>
    <tr><td>gate_scores（写 + topk 读）</td><td>[N, E]</td><td>2·N·E·2 B</td><td><b>0</b>（UB 原地）</td></tr>
    <tr><td>topk_scores（写 + div 读）</td><td>[N, K]</td><td>2·N·K·2 B</td><td><b>0</b>（UB 数组）</td></tr>
    <tr><td>kernel 发射次数</td><td>—</td><td>4</td><td><b>1</b></td></tr>
    <tr><td>中间张量流量合计</td><td>—</td><td>16·N·E + 8·N·K (B)</td><td><b>0</b></td></tr>
  </tbody>
</table>

数值例：N=4096、E=8、K=2 时，中间张量流量 ≈ 576 KB → 融合后为 0。

### 3.2 融合可行性判据（数据结构视角）

子图可融合需满足两个条件：

1. **容量条件**：中间张量按行分块后 ≤ 单核 UB（192KB）。本算子每行中间量
   = scores 行（E·4B）+ topk 暂存（K·8B），E ≤ 32 时 < 0.5KB，条件宽裕；
2. **依赖条件**：切分维度上无跨核依赖。Router 逐行独立，按 token 行（N 维）
   切分天然满足。

反例：若 E > 1024（单块超 UB）或 softmax 需跨行全局归一（跨核依赖），融合收益消失。

### 3.3 Top-K 选型：选择问题，不是排序问题

K 轮 "取最大 + 掩蔽"（selection，O(K·E)）即可解决，无需全排序（O(E·log E)）——
E ≤ 32、K ≤ 8 时 K 轮 max 就是最优选择算法族。本算子采用标量 K 轮 max，
严格 `>` 保留最小索引（与参考实现的并列处理一致）。

In [ ]:
# 查看设计文档的章节目录与流量估算节选
run('grep -n "^## " docs/design.md')

## 步骤 4：算子工程与源码走读

### 4.1 工程结构

In [ ]:
# 工程文件树（不含构建产物）
run("find src/custom_op -type f -not -path '*build_out*' -not -path '*build/*' | sort")

### 4.2 为什么是"纯标量"实现？

本算子与第 8 章 attention_custom 同构：**标量 GM 访问 + UB 普通数组**，
0 向量指令、0 队列、0 workspace。原因（本环境 910B + CANN 9.0.0 实测）：

1. 向量 Cast 不支持 BF16↔FP32（接口采用 FP16）；
2. `SyncAll()` 触发 MIX 编译模式（AIC 侧任务使向量指令空操作、双任务竞写）；
3. reduce 族指令（vcadd/vpadd/vcgadd）不产出结果，高阶库 ReduceSum 挂起；
4. 「循环内标量读取 + 向量指令混用」存在编译器缺陷（向量写 → 标量读竞态/读零）。

纯标量路线整体绕开上述失效面；代价是计算吞吐受标量流水限制（步骤 7 定量分析）。

### 4.3 Tiling 数据结构

In [ ]:
# tiling 结构体（host 填充、kernel 消费）
run("sed -n '/struct MoeRouterFusedTilingData/,/};/p' src/custom_op/op_kernel/moe_router_fused_tiling.h")

### 4.4 多核切分：为什么必须"连续块 + 64B 对齐"？

<div style="text-align: left;">
  <img src="./images/moe_tiling.svg" alt="多核行切分与输出 cache line 归属" width="780">
</div>

**关键硬件约束（实测踩坑）**：910B 上 kernel 的标量 GM 写（`SetValue`）经 L2 缓存
（line = 64B），**多核并发写同一条 line 会非确定性地丢失部分写**（核间无写一致性）。
本算子输出极小（每行仅 K·4B idx + K·2B wt），若采用跨步行进切分
（`n = coreId; n += blockDim`），各核输出按字节交错，必然共写同一 line。

**修复**：连续块切分 + 块边界 64B 对齐——每核负责连续行区间
`[c·rowsPerCore, (c+1)·rowsPerCore)`（尾核收尾到 N），
其中 `rowsPerCore` 为 `R_align = 64/gcd(2K, 64)` 的整数倍，
保证任意块边界的 idx/wt 字节偏移都落在 64B 对齐处，**每条 line 只被一个核写入**。

In [ ]:
# Host 侧 TilingFunc：切分计算（含均衡负载）
run("sed -n '/多核执行：按 token 行/,/tiling->rowsPerCore = rowsPerCore;/p' src/custom_op/op_host/moe_router_fused.cpp")

In [ ]:
# Kernel 侧：切分落地 + 逐行计算骨架（点积部分）
run("sed -n '/inline void KernelMoeRouterFused::Process/,/顺带求行内 max/p' src/custom_op/op_kernel/moe_router_fused.cpp")

In [ ]:
# Kernel 侧：K 轮 max topk + renorm 写回
run("sed -n '/---- 3) TopK/,/^}$/p' src/custom_op/op_kernel/moe_router_fused.cpp")

## 步骤 5：编译 → 打包 → 部署

### 5.1 编译并打包算子包

`build.sh` 完成：CMake preset 配置 → op_host / op_kernel 编译 → 打包为
`custom_opp_ubuntu_aarch64.run`。部署不需要 root 权限——`test/run.sh` 会把
包内容展开到 `build_out/opp_pkg`，通过 `ASCEND_CUSTOM_OPP_PATH` 环境变量加载。

In [ ]:
# 编译 + 打包（约 2-3 分钟）
run(f'source {ASCEND_HOME}/set_env.sh && cd src/custom_op && bash build.sh 2>&1 | tail -4')

## 步骤 6：正确性验证

### 6.1 比对口径

- `topk_idx`：逐元素精确匹配；允许"近似并列"翻转（|w_kernel − w_ref| < 1e-4 视为并列）；
- `topk_weights`：rtol = atol = 1e-2（FP16 量化误差，实测 max_abs ≤ 7.6e-4）。

### 6.2 单用例验证（标准形态）

In [ ]:
# 单用例：case_128_512_16_2（run.sh 自动部署算子包 + 编译测试程序）
run(f'source {ASCEND_HOME}/set_env.sh && bash src/custom_op/test/run.sh data/case_128_512_16_2')

### 6.3 8 用例一键回归

预期输出 `8/8 PASS`。注意 tiling 行中 blockDim/rowsPerCore 随形状变化：
小 N（如 16）自动退化单核，大 N 用满 40 核。

In [ ]:
# 全部 8 组边界用例回归
run(f'source {ASCEND_HOME}/set_env.sh && bash tools/run_all.sh')

## 步骤 7：性能测量与归因分析

测量方法：`test/main.cpp --bench <iters>` 使用 aclrt 事件计时（3 次预热 + N 次测量取均值）。
对照基线为步骤 2 的 4 算子序列（torch_npu）耗时。

In [ ]:
# 4 个代表性形状的性能测量（小 N / 标准 / 大 E / 大 N）
run(f'source {ASCEND_HOME}/set_env.sh && '
    'bash src/custom_op/test/run.sh data/case_16_256_8_2   --bench 100 2>&1 | grep -E "tiling|bench|PASS" && '
    'bash src/custom_op/test/run.sh data/case_128_512_16_2  --bench 100 2>&1 | grep -E "tiling|bench|PASS" && '
    'bash src/custom_op/test/run.sh data/case_1024_1024_32_4 --bench 20 2>&1 | grep -E "tiling|bench|PASS" && '
    'bash src/custom_op/test/run.sh data/case_4096_2048_8_2  --bench 10 2>&1 | grep -E "tiling|bench|PASS"')

### 7.1 结果与归因

参考实测（910B3，与你的环境应数量级一致）：

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr><th>用例 (N·D·E·K)</th><th>blockDim</th><th>fused（纯标量）</th><th>4 算子 baseline</th><th>比值</th></tr>
  </thead>
  <tbody>
    <tr><td>16·256·8·2</td><td>1</td><td>~0.34 ms</td><td>—（发射下限 ~0.23 ms）</td><td>~1.5×</td></tr>
    <tr><td>128·512·16·2</td><td>8</td><td>~2.4 ms</td><td>0.226 ms</td><td>~10× 慢</td></tr>
    <tr><td>1024·1024·32·4</td><td>32</td><td>~106 ms</td><td>0.263 ms</td><td>~400× 慢</td></tr>
    <tr><td>4096·2048·8·2</td><td>37</td><td>~61 ms</td><td>0.310 ms</td><td>~200× 慢</td></tr>
  </tbody>
</table>

**归因分析**（本章性能分析的核心结论）：

- **融合收益为真**：中间张量零落 GM（步骤 3 流量表）+ 发射 4→1。小 N（16）时
  融合版已与 4 算子序列同量级——此时两者都被发射/调度开销主导；
- **但计算吞吐由执行管线决定**：纯标量实现把计算搬到了标量流水，
  每次点积需要 O(D·E) 次标量 GM 读（有效延迟 ~50ns/次/核，第 8 章已标定），
  arithmetic intensity 过低（每字节访存仅 ~1 次标量运算），大 N 下成为瓶颈；
- **结论**：融合改变访存形态，不改变计算吞吐；要同时拿到两者，需要
  在融合骨架上换用更高吞吐的执行管线（向量化 / Cube）——这正是 12.03 实践题的方向。

## 实验总结

你已经完成了：

1. ✅ 环境检查与基线建立（4 算子序列耗时、8 组测试数据）
2. ✅ 融合设计分析（流量清零、可行性判据、Top-K 选型）
3. ✅ 纯标量融合算子源码走读（含多核共写 cache line 约束与连续块切分）
4. ✅ 编译 → 打包 → 部署（免 root，ASCEND_CUSTOM_OPP_PATH）
5. ✅ 正确性验证：单用例 + 8 用例回归全 PASS
6. ✅ 性能测量与定量归因（访存收益为真、标量吞吐是瓶颈）

**下一步**：完成 [12.03 章节实践](12.03_chapter_test.ipynb)——三道编程实践题（简单/中等/困难）+ 知识测验。
